# MFI: Modern Fortran Interfaces — GPU/CuBLAS Test

**Repo:** https://github.com/14NGiestas/mfi/tree/main

Sets up the Fortran + CUDA toolchain on Colab, builds with cuBLAS, and tests:
- Lazy init (no setup code needed)
- `MFI_USE_CUBLAS=1` env var activation
- `mfi_force_gpu` / `mfi_force_cpu` runtime switching
- CPU vs GPU correctness verification
- OpenMP thread safety

**Runtime:** Select **GPU** (T4) before running.

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Diagnose CUDA / cuBLAS availability
import os, subprocess, glob

print("=== CUDA DRIVER ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
for line in result.stdout.split('\n')[:5]:
    print(line)

print("\n=== CUDA TOOLKIT ===")
cuda_candidates = glob.glob("/usr/local/cuda*") + glob.glob("/usr/lib/cuda*")
for p in cuda_candidates:
    print(f"  Found: {p}")

print("\n=== cuBLAS libraries ===")
!ldconfig -p 2>/dev/null | grep cublas || echo "  (not in ldconfig)"
!find /usr -name "libcublas*" 2>/dev/null | head -5

print("\n=== cuBLAS headers ===")
!find /usr -name "cublas*.h" 2>/dev/null | head -5

print("\n=== nvcc ===")
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "  nvcc NOT FOUND")

In [ ]:
# 3. Install Fortran toolchain + BLAS/LAPACK
!apt-get update -qq && apt-get install -y -qq \
    gfortran libblas-dev liblapack-dev libopenblas-dev \
    python3-pip 2>&1 | tail -3
!pip install -q fypp

# Install fpm 0.13.0 (supports profiles)
!curl -sL https://github.com/fortran-lang/fpm/releases/download/v0.13.0/fpm-0.13.0-linux-x86_64-gcc-12 \
    -o /usr/local/bin/fpm && chmod +x /usr/local/bin/fpm

!echo '--- versions ---'
!gfortran --version | head -1
!fpm --version
!fypp --version

In [ ]:
# 4. Install CUDA Toolkit (headers + libraries) if missing
import subprocess, os, glob

r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
has_nvcc = r.returncode == 0

if not has_nvcc:
    print("Installing CUDA toolkit...")
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "nvidia-cuda-toolkit"],
        capture_output=True, text=True
    )
    r2 = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    if r2.returncode != 0:
        print("Trying libcublas-dev...")
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", "libcublas-dev"],
            capture_output=True, text=True
        )
    print("\n=== After install ===")
    !nvcc --version 2>&1 | head -4
else:
    print("CUDA toolkit already installed")
    !nvcc --version | head -4

# Set up paths
header_found = glob.glob("/usr/local/cuda*/include/cuda_runtime.h")
if not header_found:
    header_found = glob.glob("/usr/include/cuda_runtime.h")

if header_found:
    include_dir = os.path.dirname(header_found[0])
    cuda_base = os.path.dirname(include_dir)
    lib_dir = f"{cuda_base}/lib64"
    if not os.path.isdir(lib_dir):
        lib_dir = f"{cuda_base}/targets/x86_64-linux/lib"
    if not os.path.isdir(lib_dir):
        lib_dir = "/usr/lib/x86_64-linux-gnu"

    os.environ["CPATH"] = f"{include_dir}:{os.environ.get('CPATH', '')}"
    os.environ["LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LIBRARY_PATH', '')}"
    os.environ["LD_LIBRARY_PATH"] = f"{lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    print(f"CUDA headers: {include_dir}")
    print(f"CUDA libs:    {lib_dir}")
else:
    print("WARNING: cuda_runtime.h not found")

In [ ]:
# 5. Clone MFI
import subprocess
%cd /content
!rm -rf mfi
!git clone -b main --single-branch https://github.com/14NGiestas/mfi.git
%cd mfi
GIT_SHA = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"\nCloned commit: {GIT_SHA}")
!git log -1 --oneline

In [ ]:
# 6. Generate .f90 sources
%cd /content/mfi
!make clean 2>/dev/null
!make
print("\nSource generation complete")

In [ ]:
# 7. Set up consumer project
%cd /content
!mkdir -p consumer/app

# Create fpm.toml
import textwrap
with open('consumer/fpm.toml', 'w') as f:
    f.write(textwrap.dedent('''\
        name = "consumer"
        version = "0.1.0"
        
        [dependencies]
        mfi = { path = "../mfi", features=["cublas"] }
        
        [build]
        link = ["cublas", "cudart", "gomp"]
    '''))

# Create test program (use write() to avoid Python parsing Fortran syntax)
fortran_code = []
fortran_code.append('program test_gpu')
fortran_code.append('    use mfi_blas, only: mfi_gemm, mfi_force_gpu, mfi_force_cpu')
fortran_code.append('    use mfi_blas, only: mfi_cublas_is_active')
fortran_code.append('    use iso_fortran_env, only: real64')
fortran_code.append('    implicit none')
fortran_code.append('    real(real64) :: A(32,32), B(32,32), C_cpu(32,32), C_gpu(32,32)')
fortran_code.append('    real(real64) :: diff')
fortran_code.append('')
fortran_code.append('    ! Force GPU mode')
fortran_code.append('    call mfi_force_gpu')
fortran_code.append('    if (mfi_cublas_is_active()) then')
fortran_code.append('        print *, "GPU mode active"')
fortran_code.append('    else')
fortran_code.append('        print *, "CPU mode active"')
fortran_code.append('    end if')
fortran_code.append('')
fortran_code.append('    call random_number(A)')
fortran_code.append('    call random_number(B)')
fortran_code.append('    C_cpu = 0.0_real64')
fortran_code.append('    C_gpu = 0.0_real64')
fortran_code.append('')
fortran_code.append('    ! CPU test (via mfi_force_cpu)')
fortran_code.append('    print *, "=== CPU test ==="')
fortran_code.append('    call mfi_force_cpu')
fortran_code.append('    call mfi_gemm(A, B, C_cpu)')
fortran_code.append('    print *, "CPU done"')
fortran_code.append('')
fortran_code.append('    ! GPU test (via mfi_force_gpu)')
fortran_code.append('    print *, "=== GPU test ==="')
fortran_code.append('    call mfi_force_gpu')
fortran_code.append('    call mfi_gemm(A, B, C_gpu)')
fortran_code.append('    print *, "GPU done"')
fortran_code.append('')
fortran_code.append('    ! Verify')
fortran_code.append('    diff = maxval(abs(C_gpu - C_cpu))')
fortran_code.append('    print \'(A, ES12.4)\', "Max diff: ", diff')
fortran_code.append('    if (diff < 1e-6_real64) then')
fortran_code.append('        print *, "PASS: CPU and GPU results match"')
fortran_code.append('    else')
fortran_code.append('        print *, "FAIL: CPU/GPU mismatch"')
fortran_code.append('        error stop')
fortran_code.append('    end if')
fortran_code.append('end program')

with open('consumer/app/main.f90', 'w') as f:
    f.write('\n'.join(fortran_code) + '\n')

%cd /content/consumer

In [ ]:
# 8. Build with cuBLAS (consumer project)
%cd /content/consumer

# Set CUDA library path for fpm
import os, glob
cuda_lib = "/usr/lib/x86_64-linux-gnu"
if not glob.glob(f"{cuda_lib}/libcublas*"):
    for p in glob.glob("/usr/local/cuda*/lib64"):
        cuda_lib = p
        break
    else:
        cuda_lib = "/usr/lib/x86_64-linux-gnu"

os.environ["LIBRARY_PATH"] = f"{cuda_lib}:{os.environ.get('LIBRARY_PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["CPATH"] = f"/usr/local/cuda/include:{os.environ.get('CPATH', '')}"

print(f"Using CUDA lib path: {cuda_lib}")

!fpm build --profile cublas 2>&1

In [ ]:
# 9. Run GPU test
%cd /content/consumer
import os
os.environ["MFI_USE_CUBLAS"] = "1"

result = !fpm run --profile cublas 2>&1
print("\n".join(result))

In [ ]:
# 10. Stress test: run full MFI gemm tests on T4
%cd /content/consumer
!MFI_USE_CUBLAS=1 MFI_TEST_ELEMENTS=50000 MFI_TEST_SAMPLES=1 (cd ../mfi/; fpm test --profile cublas 2>&1 | grep gemm)

In [ ]:
# 11. Test: build WITHOUT cuBLAS — verify mfi_force_gpu/cpu are no-op stubs
%cd /content/consumer
!fpm build --profile cpu 2>&1

# The same source should compile and run on CPU without errors
result = !fpm run --profile cpu 2>&1
print("\n".join(result))